In [ ]:
#SETUP DOCKER+LANGFUSE

#!cd ~/Desktop/Mini-Project-LangFuse/langfuse
#!docker compose up -d
#http://localhost:3000/

In [ ]:
#SETUP LLM+LANGFUSE

#Import keys
import os
import json
from dotenv import load_dotenv
load_dotenv()

#Set up the LLM client
from openai import OpenAI
from google import genai
from langfuse import observe, Langfuse

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY_1"),
    base_url="https://api.groq.com/openai/v1"
)


gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
result = gemini_client.models.embed_content(
    model="models/gemini-embedding-001",
    contents="This is a test sentence."
)

#Connect LangFuse
langfuse = Langfuse()

#Models
BASELINE_MODEL = "llama-3.1-8b-instant"
NEW_MODEL = "llama-3.3-70b-versatile"

In [ ]:
#SIMPLE CHATBOT

#Prompt
SYSTEM_PROMPT = """You are a helpful healthcare assistant. You provide accurate, 
general health information to help users understand medical topics.

Rules:
- Always recommend consulting a doctor for specific medical advice.
- Be clear and concise.
- If you don't know something, say so honestly.
- Never diagnose conditions or prescribe medication.
"""

#ChatBot
@observe()
def healthcare_chatbot(user_question, model=BASELINE_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_question}
        ]
    )
    return response.choices[0].message.content

In [ ]:
#AUX FUNCTIONS (EMBEDDING AND SIMILARITY)

import numpy as np

def get_embedding(text):
    result = gemini_client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=text
    )
    
    return result.embeddings[0].values

def cosine_similarity(vec_a, vec_b):
    a = np.array(vec_a)
    b = np.array(vec_b)
    
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
#EMBEDDING SIMILARITY EVALUATION

def run_embedding_evaluation(model_name):
    dataset = langfuse.get_dataset("healthcare-golden-dataset")
    
    print(f"\nEmbedding evaluation: {model_name}")
    print(f"{'='*50}\n")
    
    scores = []
    for item in dataset.items:
        print(f"Running: {item.input[:50]}...")
        
        with item.run(run_name=f"embedding-eval-{model_name}") as span:
            answer = healthcare_chatbot(item.input, model=model_name)
            
            expected_emb = get_embedding(item.expected_output)
            actual_emb = get_embedding(answer)
            similarity = cosine_similarity(expected_emb, actual_emb)
            
            span.update(input=item.input, output=answer)
            
            trace_id = get_client().get_current_trace_id()
            if trace_id:
                langfuse.create_score(
                    trace_id=trace_id,
                    name="embedding-similarity",
                    value=round(float(similarity), 4)
                )
            
            scores.append({
                "id": item.metadata.get("id", "?"),
                "category": item.metadata.get("category", "unknown"),
                "question": item.input,
                "similarity": float(similarity)
            })
            
            print(f"  Similarity: {similarity:.3f}")
    
    avg = sum(s["similarity"] for s in scores) / len(scores)
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print(f"Average similarity: {avg:.3f}")
    print(f"Min: {min(s['similarity'] for s in scores):.3f} | Max: {max(s['similarity'] for s in scores):.3f}")
    
    return scores, avg

baseline_scores, baseline_avg = run_embedding_evaluation(BASELINE_MODEL)
new_scores, new_avg = run_embedding_evaluation(NEW_MODEL)